# 🎯 ESG Greenwashing Risk Prediction - XGBoost Model Training

**Goal:** Train an XGBoost classifier to predict greenwashing risk levels (HIGH/MODERATE/LOW) based on company ESG metrics.

**Methodology:**
- Multi-class classification (3 classes)
- Feature engineering from financial & ESG data
- SHAP for explainability
- Industry-adjusted predictions

**Author:** ESG Analysis Team  
**Date:** 2024  
**Platform:** Google Colab

---
## 📦 SECTION 1: Setup & Data Loading
Install required packages and load the ESG dataset

In [ ]:
# Install required packages
!pip install xgboost lightgbm shap joblib scikit-learn pandas numpy matplotlib seaborn -q

print("✅ Packages installed successfully!")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import joblib
import shap
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("📚 Libraries imported successfully!")
print(f"XGBoost version: {xgb.__version__}")
print(f"SHAP version: {shap.__version__}")

In [ ]:
# Mount Google Drive (optional - if dataset is in Drive)
from google.colab import drive
drive.mount('/content/drive')

# Set data path
# Option 1: From Google Drive
# DATA_PATH = '/content/drive/MyDrive/ESG/data/company_esg_financial_dataset.csv'

# Option 2: Upload directly
from google.colab import files
print("📤 Please upload company_esg_financial_dataset.csv")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]

print(f"✅ Data path set: {DATA_PATH}")

In [ ]:
# Load dataset
print("📂 Loading dataset...")
df = pd.read_csv(DATA_PATH)

print(f"\n✅ Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n{'='*60}")
print("COLUMN NAMES:")
print(f"{'='*60}")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

print(f"\n{'='*60}")
print("FIRST 5 ROWS:")
print(f"{'='*60}")
display(df.head())

print(f"\n{'='*60}")
print("DATA TYPES:")
print(f"{'='*60}")
print(df.dtypes)

print(f"\n{'='*60}")
print("MISSING VALUES:")
print(f"{'='*60}")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing': missing,
    'Percentage': missing_pct
}).sort_values('Missing', ascending=False)
print(missing_df[missing_df['Missing'] > 0])

if missing_df['Missing'].sum() == 0:
    print("\n✅ No missing values!")

print(f"\n{'='*60}")
print("BASIC STATISTICS:")
print(f"{'='*60}")
display(df.describe())

---
## 🔧 SECTION 2: Feature Engineering
Create predictive features from the raw ESG & financial data

In [ ]:
print("🔧 Starting Feature Engineering...\n")

# Create a copy for feature engineering
df_fe = df.copy()

# ============================================================
# BASE ESG FEATURES
# ============================================================
print("1️⃣ Creating base ESG features...")

# Primary ESG score
df_fe['esg_score'] = df_fe['ESG_Overall']

# ESG component scores (if available)
if 'Environmental_Score' in df_fe.columns:
    df_fe['env_score'] = df_fe['Environmental_Score']
if 'Social_Score' in df_fe.columns:
    df_fe['social_score'] = df_fe['Social_Score']
if 'Governance_Score' in df_fe.columns:
    df_fe['gov_score'] = df_fe['Governance_Score']

print("   ✅ Base ESG features created")

# ============================================================
# FINANCIAL FEATURES
# ============================================================
print("\n2️⃣ Creating financial features...")

# Revenue (log-transformed to handle scale)
df_fe['revenue_log'] = np.log1p(df_fe['Revenue'])  # log(1 + Revenue) to handle 0s

# Profit margin
df_fe['profit_margin'] = df_fe['ProfitMargin']

# Market cap (if available)
if 'MarketCap' in df_fe.columns:
    df_fe['market_cap_log'] = np.log1p(df_fe['MarketCap'])

print("   ✅ Financial features created")

# ============================================================
# ENVIRONMENTAL EFFICIENCY FEATURES
# ============================================================
print("\n3️⃣ Creating environmental efficiency features...")

# Carbon intensity (emissions per dollar of revenue)
df_fe['carbon_intensity'] = df_fe['CarbonEmissions'] / (df_fe['Revenue'] + 1e-6)  # Avoid division by zero

# Water efficiency (water usage per dollar of revenue)
df_fe['water_efficiency'] = df_fe['WaterUsage'] / (df_fe['Revenue'] + 1e-6)

# Energy efficiency (energy consumption per dollar of revenue)
df_fe['energy_efficiency'] = df_fe['EnergyConsumption'] / (df_fe['Revenue'] + 1e-6)

# Replace infinities with NaN and then fill with median
df_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
df_fe['carbon_intensity'].fillna(df_fe['carbon_intensity'].median(), inplace=True)
df_fe['water_efficiency'].fillna(df_fe['water_efficiency'].median(), inplace=True)
df_fe['energy_efficiency'].fillna(df_fe['energy_efficiency'].median(), inplace=True)

print("   ✅ Environmental efficiency features created")

# ============================================================
# INDUSTRY ENCODING
# ============================================================
print("\n4️⃣ Encoding categorical features...")

# Label encode industry
le_industry = LabelEncoder()
df_fe['industry_encoded'] = le_industry.fit_transform(df_fe['Industry'])

# Save industry mapping for later
industry_mapping = dict(zip(le_industry.classes_, le_industry.transform(le_industry.classes_)))
print(f"   Industries found: {len(industry_mapping)}")
print(f"   Sample mapping: {dict(list(industry_mapping.items())[:5])}")

print("   ✅ Industry encoding complete")

# ============================================================
# ADDITIONAL DERIVED FEATURES
# ============================================================
print("\n5️⃣ Creating additional derived features...")

# ESG momentum (if historical data available)
# For now, use proxy: deviation from industry average
industry_avg_esg = df_fe.groupby('Industry')['esg_score'].transform('mean')
df_fe['esg_vs_industry'] = df_fe['esg_score'] - industry_avg_esg

# Revenue growth proxy (if we had time series, this would be YoY growth)
# For now, use revenue relative to industry median
industry_median_revenue = df_fe.groupby('Industry')['Revenue'].transform('median')
df_fe['revenue_vs_industry'] = df_fe['Revenue'] / (industry_median_revenue + 1e-6)

# ESG disclosure quality proxy (number of non-zero ESG metrics)
esg_cols = ['CarbonEmissions', 'WaterUsage', 'EnergyConsumption']
df_fe['esg_disclosure_count'] = df_fe[esg_cols].apply(lambda row: (row > 0).sum(), axis=1)

print("   ✅ Additional features created")

# ============================================================
# CREATE TARGET VARIABLE (RISK LEVEL)
# ============================================================
print("\n6️⃣ Creating target variable (risk_level)...")

def categorize_risk(esg_score):
    """Convert ESG score to risk level"""
    if esg_score < 40:
        return 2  # HIGH risk
    elif esg_score < 70:
        return 1  # MODERATE risk
    else:
        return 0  # LOW risk

df_fe['risk_level'] = df_fe['esg_score'].apply(categorize_risk)

# Create readable labels
risk_labels = {0: 'LOW', 1: 'MODERATE', 2: 'HIGH'}
df_fe['risk_label'] = df_fe['risk_level'].map(risk_labels)

print("   ✅ Target variable created")
print("\n   Risk Distribution:")
risk_dist = df_fe['risk_label'].value_counts().sort_index()
for label, count in risk_dist.items():
    pct = count / len(df_fe) * 100
    print(f"      {label:8s}: {count:5d} ({pct:5.1f}%)")

# ============================================================
# SUMMARY
# ============================================================
print(f"\n{'='*60}")
print("FEATURE ENGINEERING SUMMARY")
print(f"{'='*60}")
print(f"Total features created: {len(df_fe.columns)}")
print(f"Original columns: {len(df.columns)}")
print(f"New features: {len(df_fe.columns) - len(df.columns)}")
print(f"\n✅ Feature Engineering Complete!")

# Display sample
feature_cols = [
    'esg_score', 'revenue_log', 'profit_margin', 
    'carbon_intensity', 'water_efficiency', 'energy_efficiency',
    'industry_encoded', 'esg_vs_industry', 'risk_level', 'risk_label'
]
print(f"\nSample of engineered features:")
display(df_fe[feature_cols].head(10))

In [ ]:
# Visualize feature distributions
print("📊 Visualizing feature distributions...\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Feature Distributions by Risk Level', fontsize=16, fontweight='bold')

features_to_plot = [
    'esg_score', 'carbon_intensity', 'water_efficiency',
    'energy_efficiency', 'profit_margin', 'esg_vs_industry'
]

for idx, feature in enumerate(features_to_plot):
    ax = axes[idx // 3, idx % 3]
    
    for risk_level, color in zip(['LOW', 'MODERATE', 'HIGH'], ['green', 'orange', 'red']):
        data = df_fe[df_fe['risk_label'] == risk_level][feature]
        ax.hist(data, alpha=0.5, label=risk_level, bins=30, color=color)
    
    ax.set_xlabel(feature.replace('_', ' ').title())
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Visualizations complete!")

---
## 🔀 SECTION 3: Train-Test Split
Split data into training and testing sets with stratification

In [ ]:
print("🔀 Preparing train-test split...\n")

# Define feature columns (exclude target and metadata)
exclude_cols = [
    'Company', 'Industry', 'risk_level', 'risk_label',
    'ESG_Overall', 'Environmental_Score', 'Social_Score', 'Governance_Score'
]

feature_cols = [col for col in df_fe.columns if col not in exclude_cols]

# Select only numeric features
X = df_fe[feature_cols].select_dtypes(include=[np.number])
y = df_fe['risk_level']

print(f"Features selected: {X.shape[1]}")
print(f"Feature names:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nTarget distribution:")
print(y.value_counts().sort_index())

# Stratified split (80% train, 20% test)
print(f"\n{'='*60}")
print("Performing stratified train-test split (80-20)...")
print(f"{'='*60}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Maintain class distribution
)

print(f"\n✅ Split complete!")
print(f"\nTraining set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Class distribution:")
for label, count in y_train.value_counts().sort_index().items():
    print(f"    {risk_labels[label]:8s}: {count:5d} ({count/len(y_train)*100:5.1f}%)")

print(f"\nTest set:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")
print(f"  Class distribution:")
for label, count in y_test.value_counts().sort_index().items():
    print(f"    {risk_labels[label]:8s}: {count:5d} ({count/len(y_test)*100:5.1f}%)")

# Check for any NaN values
print(f"\nData quality check:")
print(f"  X_train NaN count: {X_train.isna().sum().sum()}")
print(f"  X_test NaN count: {X_test.isna().sum().sum()}")

if X_train.isna().sum().sum() > 0:
    print(f"  ⚠️ Filling NaN values with median...")
    X_train = X_train.fillna(X_train.median())
    X_test = X_test.fillna(X_train.median())  # Use train median for test
    print(f"  ✅ NaN values handled")

---
## 🚀 SECTION 4: XGBoost Model Training
Train the XGBoost classifier with optimized hyperparameters

In [ ]:
print("🚀 Training XGBoost classifier...\n")

# Initialize XGBoost classifier
print("Model configuration:")
model_config = {
    'n_estimators': 100,
    'max_depth': 5,
    'learning_rate': 0.1,
    'random_state': 42,
    'objective': 'multi:softprob',  # Multi-class classification
    'num_class': 3,  # LOW, MODERATE, HIGH
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',  # Faster training
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

for key, value in model_config.items():
    print(f"  {key:20s}: {value}")

model = xgb.XGBClassifier(**model_config)

# Train with evaluation set
print(f"\n{'='*60}")
print("Training in progress...")
print(f"{'='*60}\n")

eval_set = [(X_train, y_train), (X_test, y_test)]

model.fit(
    X_train, y_train,
    eval_set=eval_set,
    verbose=10  # Print every 10 iterations
)

print(f"\n{'='*60}")
print("✅ Training complete!")
print(f"{'='*60}")

# Get training history
results = model.evals_result()

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
epochs = len(results['validation_0']['mlogloss'])
x_axis = range(0, epochs)

ax1.plot(x_axis, results['validation_0']['mlogloss'], label='Train')
ax1.plot(x_axis, results['validation_1']['mlogloss'], label='Test')
ax1.legend()
ax1.set_ylabel('Log Loss')
ax1.set_xlabel('Iteration')
ax1.set_title('XGBoost Training Curve')
ax1.grid(alpha=0.3)

# Feature importance
importance = model.feature_importances_
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importance
}).sort_values('importance', ascending=False).head(10)

ax2.barh(feature_importance['feature'], feature_importance['importance'])
ax2.set_xlabel('Importance')
ax2.set_title('Top 10 Feature Importances')
ax2.invert_yaxis()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Top 10 Most Important Features:")
print(feature_importance.to_string(index=False))

---
## 📊 SECTION 5: Model Evaluation
Comprehensive evaluation with multiple metrics

In [ ]:
print("📊 Evaluating model performance...\n")

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

# ============================================================
# BASIC METRICS
# ============================================================
print(f"{'='*60}")
print("CLASSIFICATION METRICS")
print(f"{'='*60}\n")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")

# ============================================================
# CONFUSION MATRIX
# ============================================================
print(f"\n{'='*60}")
print("CONFUSION MATRIX")
print(f"{'='*60}\n")

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual LOW', 'Actual MODERATE', 'Actual HIGH'],
    columns=['Pred LOW', 'Pred MODERATE', 'Pred HIGH']
)
print(cm_df)

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['LOW', 'MODERATE', 'HIGH'],
            yticklabels=['LOW', 'MODERATE', 'HIGH'])
plt.title('Confusion Matrix - ESG Risk Prediction', fontsize=14, fontweight='bold')
plt.ylabel('Actual Risk Level')
plt.xlabel('Predicted Risk Level')
plt.show()

# ============================================================
# PER-CLASS METRICS
# ============================================================
print(f"\n{'='*60}")
print("DETAILED CLASSIFICATION REPORT")
print(f"{'='*60}\n")

report = classification_report(
    y_test, y_pred,
    target_names=['LOW', 'MODERATE', 'HIGH'],
    digits=4
)
print(report)

# ============================================================
# PROBABILITY CALIBRATION
# ============================================================
print(f"{'='*60}")
print("PREDICTION CONFIDENCE ANALYSIS")
print(f"{'='*60}\n")

max_probas = y_pred_proba.max(axis=1)
print(f"Average prediction confidence: {max_probas.mean():.4f}")
print(f"Median prediction confidence:  {np.median(max_probas):.4f}")
print(f"Min prediction confidence:     {max_probas.min():.4f}")
print(f"Max prediction confidence:     {max_probas.max():.4f}")

# Confidence distribution
plt.figure(figsize=(10, 5))
plt.hist(max_probas, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Maximum Predicted Probability')
plt.ylabel('Frequency')
plt.title('Distribution of Prediction Confidence', fontsize=14, fontweight='bold')
plt.axvline(max_probas.mean(), color='red', linestyle='--', label=f'Mean: {max_probas.mean():.3f}')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ============================================================
# CROSS-VALIDATION
# ============================================================
print(f"\n{'='*60}")
print("CROSS-VALIDATION (5-Fold Stratified)")
print(f"{'='*60}\n")

cv_scores = cross_val_score(
    model, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy'
)

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

print(f"\n✅ Model evaluation complete!")

---
## 🔍 SECTION 6: SHAP Explainability Analysis
Understand model predictions with SHAP (SHapley Additive exPlanations)

In [ ]:
print("🔍 Generating SHAP explanations...\n")
print("⏳ This may take 1-2 minutes for large datasets...\n")

# Create SHAP explainer
# Use TreeExplainer for tree-based models (much faster than KernelExplainer)
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for test set
# Use a sample if dataset is very large
sample_size = min(1000, len(X_test))
X_test_sample = X_test.sample(n=sample_size, random_state=42)

print(f"Computing SHAP values for {sample_size} test samples...")
shap_values = explainer.shap_values(X_test_sample)

print(f"✅ SHAP values computed!\n")
print(f"Shape: {np.array(shap_values).shape}")
print(f"  (num_classes, num_samples, num_features)\n")

# ============================================================
# SHAP SUMMARY PLOT (Global Feature Importance)
# ============================================================
print(f"{'='*60}")
print("SHAP SUMMARY PLOT - Overall Feature Impact")
print(f"{'='*60}\n")

# Summary plot for each class
class_names = ['LOW Risk', 'MODERATE Risk', 'HIGH Risk']

for class_idx, class_name in enumerate(class_names):
    print(f"\n📊 {class_name} Class:")
    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values[class_idx],
        X_test_sample,
        plot_type="bar",
        show=False,
        max_display=10
    )
    plt.title(f'Top 10 Features for {class_name} Prediction', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ============================================================
# SHAP BEESWARM PLOT (Feature Impact Distribution)
# ============================================================
print(f"\n{'='*60}")
print("SHAP BEESWARM PLOT - Feature Value Impact")
print(f"{'='*60}\n")
print("This shows how feature values (color) affect predictions (x-axis)\n")

for class_idx, class_name in enumerate(class_names):
    print(f"📊 {class_name} Class:")
    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values[class_idx],
        X_test_sample,
        plot_type="dot",
        show=False,
        max_display=10
    )
    plt.title(f'Feature Impact Distribution - {class_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ============================================================
# TOP 10 FEATURES TABLE
# ============================================================
print(f"\n{'='*60}")
print("TOP 10 MOST IMPORTANT FEATURES (By SHAP)")
print(f"{'='*60}\n")

for class_idx, class_name in enumerate(class_names):
    # Calculate mean absolute SHAP value for each feature
    mean_shap = np.abs(shap_values[class_idx]).mean(axis=0)
    
    feature_importance_shap = pd.DataFrame({
        'Feature': X_test_sample.columns,
        'Mean |SHAP|': mean_shap
    }).sort_values('Mean |SHAP|', ascending=False).head(10)
    
    print(f"\n{class_name}:")
    print(feature_importance_shap.to_string(index=False))

# ============================================================
# SAMPLE PREDICTION EXPLANATION
# ============================================================
print(f"\n{'='*60}")
print("EXAMPLE: Single Prediction Explanation")
print(f"{'='*60}\n")

# Pick a sample prediction
sample_idx = 0
sample = X_test_sample.iloc[sample_idx]
actual_class = y_test.iloc[X_test_sample.index[sample_idx]]
predicted_class = model.predict(sample.values.reshape(1, -1))[0]
pred_proba = model.predict_proba(sample.values.reshape(1, -1))[0]

print(f"Sample #{sample_idx}:")
print(f"  Actual Risk:    {risk_labels[actual_class]}")
print(f"  Predicted Risk: {risk_labels[predicted_class]}")
print(f"  Confidence: {pred_proba.max():.2%}")
print(f"\n  Probability breakdown:")
for idx, prob in enumerate(pred_proba):
    print(f"    {risk_labels[idx]:8s}: {prob:.2%}")

# Force plot for this prediction
print(f"\n📊 SHAP Force Plot (shows feature contributions):")
shap.initjs()
shap.force_plot(
    explainer.expected_value[predicted_class],
    shap_values[predicted_class][sample_idx],
    sample,
    matplotlib=True
)
plt.tight_layout()
plt.show()

print(f"\n✅ SHAP analysis complete!")

---
## 💾 SECTION 7: Save Model & Artifacts
Export trained model and supporting data

In [ ]:
print("💾 Saving model and artifacts...\n")

# Create artifacts dictionary
model_artifacts = {
    'model': model,
    'feature_names': list(X_train.columns),
    'label_encoder': le_industry,
    'risk_labels': risk_labels,
    'model_config': model_config,
    'performance_metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1)
    },
    'training_date': pd.Timestamp.now().isoformat(),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'num_features': len(X_train.columns)
}

# Save model using joblib (recommended for scikit-learn/XGBoost)
model_filename = 'xgboost_risk_model.pkl'
joblib.dump(model_artifacts, model_filename)

print(f"✅ Model saved: {model_filename}")
print(f"   File size: {os.path.getsize(model_filename) / 1024:.2f} KB\n")

# Also save just the model (smaller file)
model_only_filename = 'xgboost_model_only.pkl'
joblib.dump(model, model_only_filename)
print(f"✅ Model-only saved: {model_only_filename}")
print(f"   File size: {os.path.getsize(model_only_filename) / 1024:.2f} KB\n")

# Save feature names as JSON
import json
feature_metadata = {
    'feature_names': list(X_train.columns),
    'num_features': len(X_train.columns),
    'risk_labels': {int(k): v for k, v in risk_labels.items()},
    'performance': model_artifacts['performance_metrics']
}

with open('model_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print(f"✅ Metadata saved: model_metadata.json\n")

# ============================================================
# DOWNLOAD FILES (for Colab)
# ============================================================
print(f"{'='*60}")
print("DOWNLOAD OPTIONS")
print(f"{'='*60}\n")

print("Option 1: Download to local machine")
print("  Run the cell below to download files\n")

print("Option 2: Save to Google Drive")
print("  Files will be saved to /content/drive/MyDrive/ESG_Models/\n")

# Save to Google Drive
try:
    drive_path = '/content/drive/MyDrive/ESG_Models/'
    os.makedirs(drive_path, exist_ok=True)
    
    import shutil
    shutil.copy(model_filename, drive_path + model_filename)
    shutil.copy(model_only_filename, drive_path + model_only_filename)
    shutil.copy('model_metadata.json', drive_path + 'model_metadata.json')
    
    print(f"✅ Files saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"⚠️ Could not save to Drive: {e}")
    print("   Use download option below instead")

print(f"\n✅ All artifacts saved successfully!")

In [ ]:
# Download files to local machine
from google.colab import files

print("📥 Downloading files...\n")

files.download(model_filename)
print(f"  ✅ Downloaded: {model_filename}")

files.download(model_only_filename)
print(f"  ✅ Downloaded: {model_only_filename}")

files.download('model_metadata.json')
print(f"  ✅ Downloaded: model_metadata.json")

print("\n✅ All files downloaded!")

---
## 🧪 SECTION 8: Test Prediction Function
Create and test a production-ready prediction function

In [ ]:
def predict_risk(company_data: dict, model_path: str = 'xgboost_risk_model.pkl') -> dict:
    """
    Production-ready risk prediction function
    
    Args:
        company_data: Dictionary with company features
            Required keys: esg_score, revenue_log, profit_margin,
                         carbon_intensity, water_efficiency, energy_efficiency,
                         industry_encoded, esg_vs_industry
        model_path: Path to saved model file
    
    Returns:
        Dictionary with:
            - risk_level: str (HIGH/MODERATE/LOW)
            - confidence: float (0-1)
            - probabilities: dict {HIGH: X, MODERATE: Y, LOW: Z}
            - feature_importance: dict (top 5 features)
    """
    
    # Load model artifacts
    artifacts = joblib.load(model_path)
    model = artifacts['model']
    feature_names = artifacts['feature_names']
    risk_labels = artifacts['risk_labels']
    
    # Create feature vector in correct order
    features = []
    for feature_name in feature_names:
        if feature_name in company_data:
            features.append(company_data[feature_name])
        else:
            # Use default value (median) if feature missing
            features.append(0.0)
            print(f"⚠️ Missing feature: {feature_name}, using default")
    
    # Convert to numpy array
    X = np.array(features).reshape(1, -1)
    
    # Predict
    pred_class = model.predict(X)[0]
    pred_proba = model.predict_proba(X)[0]
    
    # Format result
    result = {
        'risk_level': risk_labels[pred_class],
        'confidence': float(pred_proba.max()),
        'probabilities': {
            'LOW': float(pred_proba[0]),
            'MODERATE': float(pred_proba[1]),
            'HIGH': float(pred_proba[2])
        },
        'numeric_prediction': int(pred_class)
    }
    
    # Add feature importance (top 5)
    feature_importances = model.feature_importances_
    top_features = sorted(
        zip(feature_names, feature_importances),
        key=lambda x: x[1],
        reverse=True
    )[:5]
    
    result['top_features'] = {
        name: float(importance) for name, importance in top_features
    }
    
    return result

print("✅ Prediction function defined!")
print("\nFunction signature:")
print("  predict_risk(company_data: dict, model_path: str) -> dict")

In [ ]:
print("🧪 Testing prediction function...\n")

# ============================================================
# TEST CASE 1: Low ESG Score (HIGH RISK)
# ============================================================
print(f"{'='*60}")
print("TEST CASE 1: Low ESG Performance (Expected: HIGH RISK)")
print(f"{'='*60}\n")

test_company_1 = {
    'esg_score': 35,  # Low ESG score
    'revenue_log': 10.0,
    'profit_margin': 15.0,
    'carbon_intensity': 500.0,  # High emissions
    'water_efficiency': 80.0,
    'energy_efficiency': 120.0,
    'industry_encoded': 5,
    'esg_vs_industry': -15.0,  # Below industry average
    'revenue_vs_industry': 1.2,
    'esg_disclosure_count': 2
}

result_1 = predict_risk(test_company_1)

print("Input Company Profile:")
for key, value in test_company_1.items():
    print(f"  {key:25s}: {value}")

print(f"\nPrediction Results:")
print(f"  Risk Level:  {result_1['risk_level']} {'✅' if result_1['risk_level'] == 'HIGH' else '❌'}")
print(f"  Confidence:  {result_1['confidence']:.2%}")
print(f"\n  Probability Breakdown:")
for level, prob in result_1['probabilities'].items():
    bar = '█' * int(prob * 50)
    print(f"    {level:8s}: {prob:6.2%} {bar}")

print(f"\n  Top 5 Influential Features:")
for feature, importance in result_1['top_features'].items():
    print(f"    {feature:25s}: {importance:.4f}")

# ============================================================
# TEST CASE 2: High ESG Score (LOW RISK)
# ============================================================
print(f"\n{'='*60}")
print("TEST CASE 2: High ESG Performance (Expected: LOW RISK)")
print(f"{'='*60}\n")

test_company_2 = {
    'esg_score': 85,  # High ESG score
    'revenue_log': 12.0,
    'profit_margin': 22.0,
    'carbon_intensity': 50.0,  # Low emissions
    'water_efficiency': 15.0,
    'energy_efficiency': 25.0,
    'industry_encoded': 3,
    'esg_vs_industry': 20.0,  # Above industry average
    'revenue_vs_industry': 1.5,
    'esg_disclosure_count': 3
}

result_2 = predict_risk(test_company_2)

print("Input Company Profile:")
for key, value in test_company_2.items():
    print(f"  {key:25s}: {value}")

print(f"\nPrediction Results:")
print(f"  Risk Level:  {result_2['risk_level']} {'✅' if result_2['risk_level'] == 'LOW' else '❌'}")
print(f"  Confidence:  {result_2['confidence']:.2%}")
print(f"\n  Probability Breakdown:")
for level, prob in result_2['probabilities'].items():
    bar = '█' * int(prob * 50)
    print(f"    {level:8s}: {prob:6.2%} {bar}")

print(f"\n  Top 5 Influential Features:")
for feature, importance in result_2['top_features'].items():
    print(f"    {feature:25s}: {importance:.4f}")

# ============================================================
# TEST CASE 3: Medium ESG Score (MODERATE RISK)
# ============================================================
print(f"\n{'='*60}")
print("TEST CASE 3: Medium ESG Performance (Expected: MODERATE RISK)")
print(f"{'='*60}\n")

test_company_3 = {
    'esg_score': 55,  # Medium ESG score
    'revenue_log': 11.0,
    'profit_margin': 18.0,
    'carbon_intensity': 200.0,
    'water_efficiency': 40.0,
    'energy_efficiency': 60.0,
    'industry_encoded': 4,
    'esg_vs_industry': 0.0,  # At industry average
    'revenue_vs_industry': 1.0,
    'esg_disclosure_count': 3
}

result_3 = predict_risk(test_company_3)

print("Input Company Profile:")
for key, value in test_company_3.items():
    print(f"  {key:25s}: {value}")

print(f"\nPrediction Results:")
print(f"  Risk Level:  {result_3['risk_level']} {'✅' if result_3['risk_level'] == 'MODERATE' else '❌'}")
print(f"  Confidence:  {result_3['confidence']:.2%}")
print(f"\n  Probability Breakdown:")
for level, prob in result_3['probabilities'].items():
    bar = '█' * int(prob * 50)
    print(f"    {level:8s}: {prob:6.2%} {bar}")

print(f"\n  Top 5 Influential Features:")
for feature, importance in result_3['top_features'].items():
    print(f"    {feature:25s}: {importance:.4f}")

print(f"\n{'='*60}")
print("✅ All test cases completed!")
print(f"{'='*60}")

---
## 📋 TRAINING SUMMARY

### ✅ Completed Tasks:

1. **Data Loading** ✓
   - Loaded ESG dataset with financial metrics
   - Analyzed structure and quality

2. **Feature Engineering** ✓
   - Created 10+ predictive features
   - Environmental efficiency metrics
   - Industry-relative features

3. **Model Training** ✓
   - XGBoost classifier (100 trees, depth=5)
   - 3-class classification (HIGH/MODERATE/LOW)
   - Cross-validation performed

4. **Evaluation** ✓
   - Accuracy, Precision, Recall, F1
   - Confusion matrix
   - Per-class metrics

5. **Explainability** ✓
   - SHAP analysis
   - Feature importance rankings
   - Individual prediction explanations

6. **Model Export** ✓
   - Saved as `.pkl` file
   - Metadata JSON
   - Production-ready function

7. **Testing** ✓
   - Validated on 3 test cases
   - All risk levels tested

---

### 🚀 Next Steps:

1. **Integrate into ESG Platform:**
   ```python
   # In your Python project:
   from ml_models.xgboost_risk_model import XGBoostRiskModel
   
   model = XGBoostRiskModel()
   prediction = model.predict(analysis_data)
   ```

2. **Monitor Performance:**
   - Track prediction accuracy over time
   - Retrain quarterly with new data

3. **Improve with More Data:**
   - Add temporal features (ESG score trends)
   - Include news sentiment
   - Incorporate controversy data

---

### 📁 Output Files:

- `xgboost_risk_model.pkl` - Full model + metadata
- `xgboost_model_only.pkl` - Model only (smaller)
- `model_metadata.json` - Feature names & config

---

**Training Date:** 2026-02-05  
**Notebook Version:** 1.0  
**Platform:** Google Colab